# Ego4D Frames — 0→100 (Colab‑ready)
This notebook extracts frames for Ego4D videos and shows how to use frame indices from the sample CSVs. It includes runnable code cells with safe defaults.


In [ ]:
# Imports and environment detection
import os, csv, ast, glob, subprocess
from pathlib import Path
try:
    from google.colab import drive  # type: ignore
    ON_COLAB = True
except Exception:
    ON_COLAB = False

SAMPLES_ROOT = Path('ego4d_samples')
V2_ANN = SAMPLES_ROOT / 'v2' / 'annotations'
MAIN_JSON_CSV = SAMPLES_ROOT / 'ego4d_json.csv'
print('ON_COLAB =', ON_COLAB)
print('Samples present:', V2_ANN.exists())


In [ ]:
# Mount Drive (only on Colab)
if ON_COLAB:
    drive.mount('/content/drive', force_remount=False)
else:
    print('Not on Colab; skipping Drive mount.')


In [ ]:
# Set paths — adjust as needed
VIDEOS_DIR = '/content/drive/MyDrive/ego4d_data/videos_v2'
FRAMES_DIR = '/content/drive/MyDrive/ego4d_data/frames_v2'
os.makedirs(FRAMES_DIR, exist_ok=True)
print('VIDEOS_DIR =', VIDEOS_DIR)
print('FRAMES_DIR =', FRAMES_DIR)


In [ ]:
# Helper: parse JSON-like cells safely
def parse_jsonish(cell: str):
    t = cell.strip()
    if (t[:1] in ("'", chr(34))) and (t[-1:] in ("'", chr(34))):
        t = t[1:-1]
    try:
        return ast.literal_eval(t)
    except Exception:
        return None

# Collect video_uids referenced in sample CSVs
video_uids = set()
if V2_ANN.exists():
    for p in sorted(V2_ANN.glob('*.csv')):
        with p.open('r', encoding='utf-8') as f:
            header = f.readline().strip()
            r = csv.reader(f)
            for i, row in enumerate(r):
                if not row or len(row) != 1:
                    continue
                d = parse_jsonish(row[0])
                if isinstance(d, dict):
                    if 'video_uid' in d:
                        video_uids.add(d['video_uid'])
                    # Many files nest video_uid inside a clips array
                    if 'clips' in d and isinstance(d['clips'], list) and d['clips']:
                        vu = d['clips'][0].get('video_uid')
                        if vu: video_uids.add(vu)
                if i > 400:  # sampling limit per file
                    break
print('Found video_uids:', len(video_uids))
print('Sample:', list(sorted(video_uids))[:5])


In [ ]:
# Verify videos exist in VIDEOS_DIR
existing, missing = [], []
for vu in sorted(list(video_uids))[:50]:  # check a subset for speed
    mp4 = os.path.join(VIDEOS_DIR, vu + '.mp4')
    (existing if os.path.exists(mp4) else missing).append(vu)
print('Existing (subset):', len(existing))
print('Missing  (subset):', len(missing))
if missing:
    print('Example missing uid:', missing[0])


In [ ]:
# Helper: frame path builder and preview util
from PIL import Image
import matplotlib.pyplot as plt

def frame_path(video_uid: str, frame_number: int, frames_dir: str = FRAMES_DIR, pad: int = 7, ext: str = 'jpg'):
    return os.path.join(frames_dir, video_uid, str(frame_number).zfill(pad) + '.' + ext)

def show_image(path: str, title: str = ''):
    img = Image.open(path).convert('RGB')
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')
    plt.show()


In [ ]:
# Extract frames with ffmpeg (toggle RUN_EXTRACTION)
RUN_EXTRACTION = False   # set True to run extraction
VIDEO_UIDS_TO_EXTRACT = list(sorted(video_uids))[:1]  # pick a small set first

if RUN_EXTRACTION:
    assert ON_COLAB, 'Extraction expects ffmpeg on Colab/Unix-like runtime'
    for vu in VIDEO_UIDS_TO_EXTRACT:
        src = os.path.join(VIDEOS_DIR, vu + '.mp4')
        out_dir = os.path.join(FRAMES_DIR, vu)
        os.makedirs(out_dir, exist_ok=True)
        cmd = ['ffmpeg', '-y', '-i', src, '-start_number', '0', os.path.join(out_dir, '%07d.jpg')]
        print('Extracting:', vu)
        r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        print('
'.join(r.stdout.splitlines()[-5:]))
else:
    print('Set RUN_EXTRACTION=True to extract frames for:', VIDEO_UIDS_TO_EXTRACT[:3])


In [ ]:
# Preview a frame if available
previewed = False
for vu in sorted(video_uids):
    pattern = os.path.join(FRAMES_DIR, vu, '*.jpg')
    imgs = sorted(glob.glob(pattern))
    if imgs:
        show_image(imgs[min(10, len(imgs)-1)], title=f'{vu}')
        previewed = True
        break
if not previewed:
    print('No frames found under FRAMES_DIR. Extract first, then re-run this cell.')


In [ ]:
# Optional: compute last frame index using ego4d_samples/ego4d_json.csv
def get_num_frames_from_main(video_uid: str):
    if not MAIN_JSON_CSV.exists():
        return None
    with MAIN_JSON_CSV.open('r', encoding='utf-8') as f:
        header = f.readline()
        for row in csv.reader(f):
            d = parse_jsonish(row[0])
            if isinstance(d, dict) and d.get('video_uid') == video_uid:
                vm = d.get('video_metadata') or {}
                return vm.get('num_frames')
    return None

for vu in sorted(video_uids)[:5]:
    nf = get_num_frames_from_main(vu)
    if nf:
        p = frame_path(vu, int(nf)-1)
        print('Last-frame path (may or may not exist yet):', p)
        break


In [ ]:
# Report: where "frame" fields appear in v2 sample CSVs
def frame_keys_report(max_rows=60):
    report = {}
    if not V2_ANN.exists():
        return report
    for p in sorted(V2_ANN.glob('*.csv')):
        keys = set()
        with p.open('r', encoding='utf-8') as f:
            header = f.readline().strip()
            for i, row in enumerate(csv.reader(f)):
                if not row or len(row)!=1:
                    continue
                d = parse_jsonish(row[0])
                if isinstance(d, dict):
                    for k, v in d.items():
                        if 'frame' in k.lower(): keys.add(k)
                        if isinstance(v, dict):
                            for kk in v.keys():
                                if 'frame' in kk.lower(): keys.add(f'{k}.{kk}')
                        elif isinstance(v, list) and v and isinstance(v[0], dict):
                            for kk in v[0].keys():
                                if 'frame' in kk.lower(): keys.add(f'{k}[].{kk}')
                if i>max_rows:
                    break
        if keys:
            report[str(p)] = sorted(keys)
    return report

rep = frame_keys_report()
for path, keys in rep.items():
    print('FILE:', path)
    for k in keys:
        print('  -', k)


Done. Next: set RUN_EXTRACTION=True to extract frames for a small subset, then plug FRAMES_DIR and frame_path() into your Starter notebook visualization.
